In [3]:
import mysql.connector
import pandas as pd


conn = mysql.connector.connect(
    host="127.0.0.1",
    user="root",
    password="Kratos@123",
    database="gow_armors"
)

cursor = conn.cursor()

cursor.execute("SELECT Berserker FROM final_recommendations")
cursor.fetchall()

rows = cursor.fetchall()
columns = [column[0] for column in cursor.description]

armor_df = pd.DataFrame(rows, columns=columns)

armor_df.head()



,Berserker


In [ ]:
armor_df.columns.tolist()

In [ ]:
stats = ["Strength", "Defence", "Runic", "Vitality", "Cooldown", "Luck"]

for stat in stats:
    armor_df[stat + "_score"] = (
        armor_df[stat] - armor_df[stat].min()
    ) / (armor_df[stat].max() - armor_df[stat].min())

In [ ]:
armor_df.columns.tolist()

In [ ]:
armor_df["Score"] = (
    armor_df["Strength_score"] * 0.20 +
    armor_df["Defence_score"] * 0.25 +
    armor_df["Runic_score"] * 0.15 +
    armor_df["Vitality_score"] * 0.15 +
    armor_df["Cooldown_score"] * 0.15 +
    armor_df["Luck_score"] * 0.10
)

In [ ]:
armor_df[
    ["Title", "Category", "Score"]
].sort_values(
    by="Score",
    ascending=False
).head(10)

In [ ]:
berserker_weights = {
    "Strength": 0.25,
    "Defence": 0.30,
    "Runic": 0.10,
    "Vitality": 0.20,
    "Cooldown": 0.10,
    "Luck": 0.05
}

In [ ]:
armor_df["Berserker_Score"] = (
    armor_df["Strength_score"] * berserker_weights["Strength"] +
    armor_df["Defence_score"] * berserker_weights["Defence"] +
    armor_df["Runic_score"] * berserker_weights["Runic"] +
    armor_df["Vitality_score"] * berserker_weights["Vitality"] +
    armor_df["Cooldown_score"] * berserker_weights["Cooldown"] +
    armor_df["Luck_score"] * berserker_weights["Luck"]
)

In [ ]:
best_by_category = (
    armor_df
    .sort_values("Score", ascending=False)
    .groupby("Category")
    .first()
)

best_by_category[["Title", "Score"]]

In [ ]:
print("Recommended Armor Build:")
print()

for category, row in best_by_category.iterrows():
    print(category, ":", row["Title"], "| Score:", round(row["Score"], 3))

In [ ]:
build_score = best_by_category["Score"].mean()

print("Complete Armor Build Score:", build_score)

In [ ]:
cursor.execute("""
CREATE TABLE berserkers (
    ID INT AUTO_INCREMENT PRIMARY KEY,
    Name VARCHAR(100),
    Defence_Weight DECIMAL(4,3),
    Strength_Weight DECIMAL(4,3),
    Runic_Weight DECIMAL(4,3),
    Vitality_Weight DECIMAL(4,3),
    Cooldown_Weight DECIMAL(4,3),
    Luck_Weight DECIMAL(4,3)
)
""")

conn.commit()

print("Berserker table created!")

In [ ]:
berserkers = [
    "Fraekni the Zealous",
    "Bodvar the Fierce",
    "Starolfr the Troublesome",
    "Hjalti the Stolid",
    "Haklangr the Bearded",
    "Beigadr the Feared",
    "Svipdagr the Cold",
    "Skjothendi the Unerring",
    "Hvitserkr the Bold",
    "Skarde the Hateful",
    "Frakki the Arrogant",
    "King Hrolf Kraki"
]

for name in berserkers:
    cursor.execute("""
        INSERT INTO berserkers
        (Name, Defence_Weight, Strength_Weight, Runic_Weight,
         Vitality_Weight, Cooldown_Weight, Luck_Weight)
        VALUES (%s, 0, 0, 0, 0, 0, 0)
    """, (name,))

conn.commit()

print("12 Berserkers inserted!")

In [ ]:
cursor.execute("SELECT * FROM berserkers")
for row in cursor.fetchall():
    print(row)

In [ ]:
weight_profiles = {
    "Defence": {
        "Defence": 0.35,
        "Strength": 0.20,
        "Vitality": 0.20,
        "Runic": 0.10,
        "Cooldown": 0.10,
        "Luck": 0.05
    },

    "Damage": {
        "Strength": 0.35,
        "Runic": 0.20,
        "Defence": 0.15,
        "Vitality": 0.10,
        "Cooldown": 0.15,
        "Luck": 0.05
    },

    "Balanced": {
        "Strength": 0.20,
        "Defence": 0.20,
        "Runic": 0.15,
        "Vitality": 0.15,
        "Cooldown": 0.15,
        "Luck": 0.15
    }
}

In [ ]:
cursor.execute("SELECT * FROM berserkers")
berserker_df = pd.DataFrame(
    cursor.fetchall(),
    columns=[col[0] for col in cursor.description]
)

berserker_df

In [ ]:
def recommend_armor(weights):

    armor_df["Recommendation_Score"] = (
        armor_df["Strength_score"] * weights["Strength"] +
        armor_df["Defence_score"] * weights["Defence"] +
        armor_df["Runic_score"] * weights["Runic"] +
        armor_df["Vitality_score"] * weights["Vitality"] +
        armor_df["Cooldown_score"] * weights["Cooldown"] +
        armor_df["Luck_score"] * weights["Luck"]
    )

    best_build = (
        armor_df
        .sort_values("Recommendation_Score", ascending=False)
        .groupby("Category")
        .first()
    )

    return best_build[[
        "Title",
        "Recommendation_Score"
    ]]

In [ ]:
recommended = recommend_armor(
    weight_profiles["Defence"]
)

recommended

In [ ]:
print(
    "Build Score:",
    round(recommended["Recommendation_Score"].mean(), 3)
)

In [ ]:
def get_berserker_weights(name):

    cursor.execute("""
        SELECT
            Strength_Weight,
            Defence_Weight,
            Runic_Weight,
            Vitality_Weight,
            Cooldown_Weight,
            Luck_Weight
        FROM berserkers
        WHERE Name = %s
    """, (name,))

    result = cursor.fetchone()

    if result is None:
        print("Berserker not found!")
        return None

    weights = {
        "Strength": float(result[0]),
        "Defence": float(result[1]),
        "Runic": float(result[2]),
        "Vitality": float(result[3]),
        "Cooldown": float(result[4]),
        "Luck": float(result[5])
    }

    return weights

In [ ]:
def recommend_berserker(name):

    weights = get_berserker_weights(name)

    if weights is None:
        return None

    armor_df["Recommendation_Score"] = (
        armor_df["Strength_score"] * weights["Strength"] +
        armor_df["Defence_score"] * weights["Defence"] +
        armor_df["Runic_score"] * weights["Runic"] +
        armor_df["Vitality_score"] * weights["Vitality"] +
        armor_df["Cooldown_score"] * weights["Cooldown"] +
        armor_df["Luck_score"] * weights["Luck"]
    )

    best_build = (
        armor_df
        .sort_values("Recommendation_Score", ascending=False)
        .groupby("Category")
        .first()
    )

    return best_build[
        ["Title", "Recommendation_Score"]
    ]

In [ ]:
recommend_berserker("King Hrolf Kraki")

In [ ]:
cursor.execute("""
UPDATE berserkers
SET
    Defence_Weight = 0.35,
    Strength_Weight = 0.20,
    Runic_Weight = 0.10,
    Vitality_Weight = 0.20,
    Cooldown_Weight = 0.10,
    Luck_Weight = 0.05
WHERE Name = 'Fraekni the Zealous'
""")

conn.commit()

print("Fraekni weights updated!")

In [ ]:
cursor.execute("""
SELECT *
FROM berserkers
WHERE Name = 'Fraekni the Zealous'
""")

print(cursor.fetchone())

In [ ]:
recommend_berserker("Fraekni the Zealous")

In [ ]:
def update_berserker_weights(
    name,
    strength,
    defence,
    runic,
    vitality,
    cooldown,
    luck
):
    cursor.execute("""
        UPDATE berserkers
        SET
            Strength_Weight = %s,
            Defence_Weight = %s,
            Runic_Weight = %s,
            Vitality_Weight = %s,
            Cooldown_Weight = %s,
            Luck_Weight = %s
        WHERE Name = %s
    """, (
        strength,
        defence,
        runic,
        vitality,
        cooldown,
        luck,
        name
    ))

    conn.commit()
    print(name, "updated!")

In [ ]:
update_berserker_weights(
    "Bodvar the Fierce",
    0.25,   # Strength
    0.35,   # Defence
    0.10,   # Runic
    0.15,   # Vitality
    0.10,   # Cooldown
    0.05    # Luck
)

In [ ]:
cursor.execute("""
SELECT * FROM berserkers
WHERE Name = 'Bodvar the Fierce'
""")

print(cursor.fetchone())

In [ ]:
recommend_berserker("Bodvar the Fierce")

In [ ]:
def show_recommendation(name):

    result = recommend_berserker(name)

    if result is None:
        return

    print("===================================")
    print(" Berserker:", name)
    print("===================================")

    total_score = result["Recommendation_Score"].mean()

    for category, row in result.iterrows():
        print(
            f"{category}: {row['Title']} "
            f"(Score: {row['Recommendation_Score']:.3f})"
        )

    print("-----------------------------------")
    print(f"Build Score: {total_score:.3f}")
    print("===================================")

In [ ]:
show_recommendation("Fraekni the Zealous")

In [ ]:
cursor.execute("""
SELECT
    ID,
    Name,
    Strength_Weight,
    Defence_Weight,
    Runic_Weight,
    Vitality_Weight,
    Cooldown_Weight,
    Luck_Weight
FROM berserkers
ORDER BY ID
""")

berserker_df = pd.DataFrame(
    cursor.fetchall(),
    columns=[col[0] for col in cursor.description]
)

berserker_df

In [ ]:
weight_columns = [
    "Strength_Weight",
    "Defence_Weight",
    "Runic_Weight",
    "Vitality_Weight",
    "Cooldown_Weight",
    "Luck_Weight"
]

berserker_df["Weight_Total"] = berserker_df[weight_columns].sum(axis=1)

berserker_df[["Name", "Weight_Total"]]

# Single Test

In [ ]:
show_recommendation("Fraekni the Zealous")

Multiple Test

In [ ]:
test_berserkers = [
    "Fraekni the Zealous",
    "Bodvar the Fierce",
    "Starolfr the Troublesome",
    "Hjalti the Stolid"
]

for name in test_berserkers:
    show_recommendation(name)

Unknown Entry test

In [ ]:
show_recommendation("Test Berserker")

In [ ]:
armor_df["Skill"].value_counts()

In [ ]:
def skill_score(skill):
    if pd.isna(skill):
        return 0

    skill = skill.lower()
    score = 0

    if "damage" in skill:
        score += 0.30

    if "reduced damage" in skill:
        score += 0.25

    if "rage" in skill:
        score += 0.20

    if "stagger" in skill:
        score += 0.15

    if "health" in skill:
        score += 0.10

    if "luck" in skill:
        score += 0.10

    if "runic" in skill:
        score += 0.10

    return min(score, 1.0)

In [ ]:
armor_df["Skill_score"] = armor_df["Skill"].apply(skill_score)

In [ ]:
armor_df[[
    "Title",
    "Skill",
    "Skill_score"
]].sort_values(
    by="Skill_score",
    ascending=False
).head(10)

In [ ]:
def recommend_berserker(name):

    weights = get_berserker_weights(name)

    if weights is None:
        return None

    armor_df["Recommendation_Score"] = (
        armor_df["Strength_score"] * weights["Strength"] +
        armor_df["Defence_score"] * weights["Defence"] +
        armor_df["Runic_score"] * weights["Runic"] +
        armor_df["Vitality_score"] * weights["Vitality"] +
        armor_df["Cooldown_score"] * weights["Cooldown"] +
        armor_df["Luck_score"] * weights["Luck"] +
        armor_df["Skill_score"] * 0.10
    )

    best_build = (
        armor_df
        .sort_values("Recommendation_Score", ascending=False)
        .groupby("Category")
        .first()
    )

    return best_build[
        ["Title", "Skill", "Recommendation_Score"]
    ]

In [ ]:
recommend_berserker("Fraekni the Zealous")

In [ ]:
skill_preferences = {
    "Fraekni the Zealous": ["damage", "strength"],
    "Bodvar the Fierce": ["defence", "stagger"],
    "Starolfr the Troublesome": ["runic", "cooldown"],
    "Hjalti the Stolid": ["defence", "health"]
}

In [ ]:
def skill_match(skill, preferences):

    if pd.isna(skill):
        return 0

    skill = skill.lower()

    matches = 0

    for preference in preferences:
        if preference.lower() in skill:
            matches += 1

    if len(preferences) == 0:
        return 0

    return matches / len(preferences)

In [ ]:
armor_df["Skill_Match"] = armor_df["Skill"].apply(
    lambda x: skill_match(
        x,
        skill_preferences["Fraekni the Zealous"]
    )
)

In [ ]:
armor_df[
    ["Title", "Skill", "Skill_Match"]
].sort_values(
    "Skill_Match",
    ascending=False
).head(10)

In [ ]:
def recommend_berserker(name):

    weights = get_berserker_weights(name)

    if weights is None:
        return None

    # Berserker di skill preferences
    preferences = skill_preferences.get(name, [])

    # Har armor di skill compatibility
    armor_df["Skill_Match"] = armor_df["Skill"].apply(
        lambda x: skill_match(x, preferences)
    )

    # Final score
    armor_df["Recommendation_Score"] = (
        armor_df["Strength_score"] * weights["Strength"] +
        armor_df["Defence_score"] * weights["Defence"] +
        armor_df["Runic_score"] * weights["Runic"] +
        armor_df["Vitality_score"] * weights["Vitality"] +
        armor_df["Cooldown_score"] * weights["Cooldown"] +
        armor_df["Luck_score"] * weights["Luck"] +
        armor_df["Skill_Match"] * 0.10
    )

    # Best Chest, Wrist and Waist
    best_build = (
        armor_df
        .sort_values("Recommendation_Score", ascending=False)
        .groupby("Category")
        .first()
    )

    return best_build[
        ["Title", "Skill", "Recommendation_Score"]
    ]

In [ ]:
recommend_berserker("Fraekni the Zealous")

In [ ]:
show_recommendation("Fraekni the Zealous")

In [ ]:
def recommendation_details(name):

    weights = get_berserker_weights(name)

    if weights is None:
        return None

    preferences = skill_preferences.get(name, [])

    armor_df["Skill_Match"] = armor_df["Skill"].apply(
        lambda x: skill_match(x, preferences)
    )

    # Individual contributions
    armor_df["Strength_Contribution"] = (
        armor_df["Strength_score"] * weights["Strength"]
    )

    armor_df["Defence_Contribution"] = (
        armor_df["Defence_score"] * weights["Defence"]
    )

    armor_df["Runic_Contribution"] = (
        armor_df["Runic_score"] * weights["Runic"]
    )

    armor_df["Vitality_Contribution"] = (
        armor_df["Vitality_score"] * weights["Vitality"]
    )

    armor_df["Cooldown_Contribution"] = (
        armor_df["Cooldown_score"] * weights["Cooldown"]
    )

    armor_df["Luck_Contribution"] = (
        armor_df["Luck_score"] * weights["Luck"]
    )

    armor_df["Skill_Contribution"] = (
        armor_df["Skill_Match"] * 0.10
    )

    armor_df["Recommendation_Score"] = (
        armor_df["Strength_Contribution"] +
        armor_df["Defence_Contribution"] +
        armor_df["Runic_Contribution"] +
        armor_df["Vitality_Contribution"] +
        armor_df["Cooldown_Contribution"] +
        armor_df["Luck_Contribution"] +
        armor_df["Skill_Contribution"]
    )

    best_build = (
        armor_df
        .sort_values("Recommendation_Score", ascending=False)
        .groupby("Category")
        .first()
    )

    return best_build[
        [
            "Title",
            "Strength_Contribution",
            "Defence_Contribution",
            "Runic_Contribution",
            "Vitality_Contribution",
            "Cooldown_Contribution",
            "Luck_Contribution",
            "Skill_Contribution",
            "Recommendation_Score"
        ]
    ]

In [ ]:
recommendation_details("Fraekni the Zealous")

In [ ]:
def final_recommendation(name):

    result = recommendation_details(name)

    if result is None:
        return

    print("=" * 55)
    print("        GOD OF WAR RAGNARÖK")
    print("        ARMOR RECOMMENDATION")
    print("=" * 55)

    print(f"\nBerserker: {name}")
    print("-" * 55)

    for category, row in result.iterrows():

        print(f"\n{category}: {row['Title']}")

        print(
            f"  Strength contribution : "
            f"{row['Strength_Contribution']:.3f}"
        )

        print(
            f"  Defence contribution  : "
            f"{row['Defence_Contribution']:.3f}"
        )

        print(
            f"  Runic contribution    : "
            f"{row['Runic_Contribution']:.3f}"
        )

        print(
            f"  Vitality contribution : "
            f"{row['Vitality_Contribution']:.3f}"
        )

        print(
            f"  Cooldown contribution : "
            f"{row['Cooldown_Contribution']:.3f}"
        )

        print(
            f"  Luck contribution     : "
            f"{row['Luck_Contribution']:.3f}"
        )

        print(
            f"  Skill contribution    : "
            f"{row['Skill_Contribution']:.3f}"
        )

        print(
            f"  Final score           : "
            f"{row['Recommendation_Score']:.3f}"
        )

    build_score = result["Recommendation_Score"].mean()

    print("\n" + "-" * 55)
    print(f"Complete Build Score: {build_score:.3f}")
    print("=" * 55)

In [ ]:
final_recommendation("Fraekni the Zealous")

In [ ]:
all_recommendations = []

for name in berserker_df["Name"]:

    result = recommendation_details(name)

    if result is None:
        continue

    row = {
        "Berserker": name,
        "Chest": result.loc["Chest", "Title"] if "Chest" in result.index else None,
        "Wrist": result.loc["Wrist", "Title"] if "Wrist" in result.index else None,
        "Waist": result.loc["Waist", "Title"] if "Waist" in result.index else None,
        "Build_Score": result["Recommendation_Score"].mean()
    }

    all_recommendations.append(row)

recommendation_df = pd.DataFrame(all_recommendations)

recommendation_df

In [ ]:
recommendation_df["Build_Score"] = (
    recommendation_df["Build_Score"].round(3)
)

recommendation_df

In [ ]:
print("Missing Chest:")
print(recommendation_df["Chest"].isna().sum())

print("Missing Wrist:")
print(recommendation_df["Wrist"].isna().sum())

print("Missing Waist:")
print(recommendation_df["Waist"].isna().sum())

In [ ]:
print(
    recommendation_df[
        ["Berserker", "Chest", "Wrist", "Waist"]
    ].duplicated().sum()
)

In [ ]:
print(recommendation_df["Build_Score"].describe())

In [ ]:
print(
    recommendation_df[
        recommendation_df["Build_Score"] < 0
    ]
)

In [ ]:
recommendation_df

In [ ]:
name = input("Enter Berserker name: ")

final_recommendation(name)

In [ ]:
name = input("Enter Berserker name: ").strip()

if name in berserker_df["Name"].values:
    final_recommendation(name)
else:
    print("Berserker not found.")
    print("\nAvailable Berserkers:")
    for berserker in berserker_df["Name"]:
        print("-", berserker)

In [ ]:
print(recommendation_df)

In [ ]:
print(recommendation_df.shape)

In [ ]:
berserker_df[
    [
        "Name",
        "Strength_Weight",
        "Defence_Weight",
        "Runic_Weight",
        "Vitality_Weight",
        "Cooldown_Weight",
        "Luck_Weight"
    ]
]

In [ ]:
weight_columns = [
    "Strength_Weight",
    "Defence_Weight",
    "Runic_Weight",
    "Vitality_Weight",
    "Cooldown_Weight",
    "Luck_Weight"
]

berserker_df["Weight_Total"] = berserker_df[weight_columns].sum(axis=1)

berserker_df[
    ["Name", "Weight_Total"]
]

In [ ]:
weight_data = {
    "Starolfr the Troublesome": (0.20, 0.20, 0.20, 0.10, 0.25, 0.05),
    "Hjalti the Stolid":         (0.20, 0.35, 0.10, 0.20, 0.10, 0.05),
    "Haklangr the Bearded":      (0.25, 0.30, 0.10, 0.20, 0.10, 0.05),
    "Beigadr the Feared":        (0.25, 0.20, 0.25, 0.10, 0.15, 0.05),
    "Svipdagr the Cold":         (0.30, 0.25, 0.10, 0.15, 0.15, 0.05),
    "Skjothendi the Unerring":   (0.20, 0.20, 0.30, 0.10, 0.15, 0.05),
    "Hvitserkr the Bold":        (0.25, 0.20, 0.15, 0.15, 0.20, 0.05),
    "Skarde the Hateful":        (0.30, 0.25, 0.10, 0.15, 0.15, 0.05),
    "Frakki the Arrogant":       (0.25, 0.30, 0.10, 0.15, 0.15, 0.05),
    "King Hrolf Kraki":          (0.20, 0.25, 0.20, 0.15, 0.15, 0.05)
}

In [ ]:
for name, weights in weight_data.items():

    cursor.execute("""
        UPDATE berserkers
        SET
            Strength_Weight = %s,
            Defence_Weight = %s,
            Runic_Weight = %s,
            Vitality_Weight = %s,
            Cooldown_Weight = %s,
            Luck_Weight = %s
        WHERE Name = %s
    """, (*weights, name))

conn.commit()

print("All Berserker weights updated!")

In [ ]:
cursor.execute("""
SELECT
    Name,
    Strength_Weight,
    Defence_Weight,
    Runic_Weight,
    Vitality_Weight,
    Cooldown_Weight,
    Luck_Weight
FROM berserkers
""")

berserker_df = pd.DataFrame(
    cursor.fetchall(),
    columns=[col[0] for col in cursor.description]
)

berserker_df

In [ ]:
all_recommendations = []

for name in berserker_df["Name"]:

    result = recommendation_details(name)

    if result is None:
        continue

    row = {
        "Berserker": name,
        "Chest": result.loc["Chest", "Title"] if "Chest" in result.index else None,
        "Wrist": result.loc["Wrist", "Title"] if "Wrist" in result.index else None,
        "Waist": result.loc["Waist", "Title"] if "Waist" in result.index else None,
        "Build_Score": result["Recommendation_Score"].mean()
    }

    all_recommendations.append(row)

recommendation_df = pd.DataFrame(all_recommendations)

recommendation_df["Build_Score"] = (
    recommendation_df["Build_Score"].round(3)
)

recommendation_df

In [ ]:
recommendation_df

In [ ]:
recommendation_df["Build_Score"].describe()

clean scoring system

In [ ]:
def recommendation_details(name):

    weights = get_berserker_weights(name)

    if weights is None:
        return None

    preferences = skill_preferences.get(name, [])

    armor_df["Skill_Match"] = armor_df["Skill"].apply(
        lambda x: skill_match(x, preferences)
    )

    # 90% of final score comes from stats
    numeric_score = (
        armor_df["Strength_score"] * weights["Strength"] +
        armor_df["Defence_score"] * weights["Defence"] +
        armor_df["Runic_score"] * weights["Runic"] +
        armor_df["Vitality_score"] * weights["Vitality"] +
        armor_df["Cooldown_score"] * weights["Cooldown"] +
        armor_df["Luck_score"] * weights["Luck"]
    )

    # 10% comes from special skill
    armor_df["Recommendation_Score"] = (
        numeric_score * 0.90 +
        armor_df["Skill_Match"] * 0.10
    )

    best_build = (
        armor_df
        .sort_values("Recommendation_Score", ascending=False)
        .groupby("Category")
        .first()
    )

    return best_build[
        [
            "Title",
            "Skill",
            "Recommendation_Score"
        ]
    ]

In [ ]:
recommendation_details("Fraekni the Zealous")

In [ ]:
result = recommendation_details("Fraekni the Zealous")

build_score = result["Recommendation_Score"].mean()

print("Complete Build Score:", round(build_score, 3))


12 berserk complete engine

In [ ]:
all_recommendations = []

for name in berserker_df["Name"]:

    result = recommendation_details(name)

    if result is None:
        continue

    row = {
        "Berserker": name,
        "Chest": result.loc["Chest", "Title"],
        "Wrist": result.loc["Wrist", "Title"],
        "Waist": result.loc["Waist", "Title"],
        "Build_Score": result["Recommendation_Score"].mean()
    }

    all_recommendations.append(row)

recommendation_df = pd.DataFrame(all_recommendations)

recommendation_df["Build_Score"] = recommendation_df["Build_Score"].round(3)

recommendation_df

In [ ]:
recommendation_df[
    ["Berserker", "Chest", "Wrist", "Waist", "Build_Score"]
]

user input based backend

In [ ]:
name = input("Enter Berserker name: ")

result = recommendation_details(name)

if result is not None:

    print("\n" + "=" * 55)
    print("GOD OF WAR RAGNARÖK - RECOMMENDED BUILD")
    print("=" * 55)

    print(f"\nBerserker: {name}")

    for category, row in result.iterrows():
        print(f"\n{category}: {row['Title']}")
        print(f"Score: {row['Recommendation_Score']:.3f}")

    build_score = result["Recommendation_Score"].mean()

    print("\n" + "-" * 55)
    print(f"Complete Build Score: {build_score:.3f}")
    print("=" * 55)

filling wrong berserk name

In [ ]:
def get_berserker_names():
    return berserker_df["Name"].tolist()


def run_recommendation():

    print("\nAvailable Berserkers:")
    
    for i, name in enumerate(get_berserker_names(), start=1):
        print(f"{i}. {name}")

    choice = input("\nEnter Berserker number: ")

    if not choice.isdigit():
        print("Please enter a valid number.")
        return

    choice = int(choice)

    names = get_berserker_names()

    if choice < 1 or choice > len(names):
        print("Invalid Berserker number.")
        return

    name = names[choice - 1]

    result = recommendation_details(name)

    print("\n" + "=" * 55)
    print("GOD OF WAR RAGNARÖK - RECOMMENDED BUILD")
    print("=" * 55)

    print(f"\nBerserker: {name}")

    for category, row in result.iterrows():
        print(f"\n{category}: {row['Title']}")
        print(f"Score: {row['Recommendation_Score']:.3f}")

    build_score = result["Recommendation_Score"].mean()

    print("\n" + "-" * 55)
    print(f"Complete Build Score: {build_score:.3f}")
    print("=" * 55)

In [ ]:
run_recommendation()

In [ ]:
cursor.execute("SHOW TABLES")

for table in cursor.fetchall():
    print(table[0])

weapons module


In [ ]:
cursor.execute("SELECT * FROM axe")

axe_df = pd.DataFrame(
    cursor.fetchall(),
    columns=[col[0] for col in cursor.description]
)

axe_df.head()

scoring axe

In [ ]:
axe_stats = [
    "Strength",
    "Defence",
    "Runic",
    "Vitality",
    "Cooldown",
    "luck"
]

for stat in axe_stats:
    axe_df[stat + "_score"] = (
        axe_df[stat] - axe_df[stat].min()
    ) / (
        axe_df[stat].max() - axe_df[stat].min()
    )

axe_df.head()

In [ ]:
axe_df[
    ["Title", "Category", "Strength_score", "Runic_score", "Cooldown_score", "luck_score"]
].head(10)

axe recomendation basis

In [ ]:
def recommend_axe(name):

    weights = get_berserker_weights(name)

    if weights is None:
        return None

    axe_df["Recommendation_Score"] = (
        axe_df["Strength_score"] * weights["Strength"] +
        axe_df["Defence_score"] * weights["Defence"] +
        axe_df["Runic_score"] * weights["Runic"] +
        axe_df["Vitality_score"] * weights["Vitality"] +
        axe_df["Cooldown_score"] * weights["Cooldown"] +
        axe_df["luck_score"] * weights["Luck"]
    )

    result = axe_df.sort_values(
        "Recommendation_Score",
        ascending=False
    )

    return result[
        ["Title", "Category", "Skill", "Recommendation_Score"]
    ]

In [ ]:
recommend_axe("King Hrolf Kraki").head(10)

blades recomendation

In [ ]:
cursor.execute("SELECT * FROM blades")

blades_df = pd.DataFrame(
    cursor.fetchall(),
    columns=[col[0] for col in cursor.description]
)

blades_df.head()

In [ ]:
blade_stats = [
    "Strength",
    "Defence",
    "Runic",
    "Vitality",
    "Cooldown",
    "luck"
]

for stat in blade_stats:
    blades_df[stat + "_score"] = (
        blades_df[stat] - blades_df[stat].min()
    ) / (
        blades_df[stat].max() - blades_df[stat].min()
    )

blades_df.head()

In [ ]:
blades_df[
    [
        "Title",
        "Category",
        "Strength_score",
        "Runic_score",
        "Cooldown_score",
        "luck_score"
    ]
].head(10)

In [ ]:
def recommend_blades(name):

    weights = get_berserker_weights(name)

    if weights is None:
        return None

    blades_df["Recommendation_Score"] = (
        blades_df["Strength_score"] * weights["Strength"] +
        blades_df["Defence_score"] * weights["Defence"] +
        blades_df["Runic_score"] * weights["Runic"] +
        blades_df["Vitality_score"] * weights["Vitality"] +
        blades_df["Cooldown_score"] * weights["Cooldown"] +
        blades_df["luck_score"] * weights["Luck"]
    )

    result = blades_df.sort_values(
        "Recommendation_Score",
        ascending=False
    )

    return result[
        ["Title", "Category", "Skill", "Recommendation_Score"]
    ]

In [ ]:
recommend_blades("King Hrolf Kraki").head(10)

spear module 

In [ ]:
cursor.execute("SELECT * FROM spear")

spear_df = pd.DataFrame(
    cursor.fetchall(),
    columns=[col[0] for col in cursor.description]
)

spear_df.head()

In [ ]:
spear_stats = [
    "Strength",
    "Defence",
    "Runic",
    "Vitality",
    "Cooldown",
    "luck"
]

for stat in spear_stats:
    spear_df[stat + "_score"] = (
        spear_df[stat] - spear_df[stat].min()
    ) / (
        spear_df[stat].max() - spear_df[stat].min()
    )

spear_df.head()

In [ ]:
spear_df[
    [
        "Title",
        "Category",
        "Strength_score",
        "Defence_score",
        "Runic_score",
        "Vitality_score",
        "Cooldown_score",
        "luck_score"
    ]
].head(10)

In [ ]:
def recommend_spear(name):

    weights = get_berserker_weights(name)

    if weights is None:
        return None

    spear_df["Recommendation_Score"] = (
        spear_df["Strength_score"] * weights["Strength"] +
        spear_df["Defence_score"] * weights["Defence"] +
        spear_df["Runic_score"] * weights["Runic"] +
        spear_df["Vitality_score"] * weights["Vitality"] +
        spear_df["Cooldown_score"] * weights["Cooldown"] +
        spear_df["luck_score"] * weights["Luck"]
    )

    result = spear_df.sort_values(
        "Recommendation_Score",
        ascending=False
    )

    return result[
        ["Title", "Category", "Skill", "Recommendation_Score"]
    ]

In [ ]:
recommend_spear("King Hrolf Kraki").head(10)

sheilds


In [ ]:
cursor.execute("SELECT * FROM shield")

shield_df = pd.DataFrame(
    cursor.fetchall(),
    columns=[col[0] for col in cursor.description]
)

shield_df.head()

In [ ]:
shield_stats = [
    "Strength",
    "Defence",
    "Runic",
    "Vitality",
    "Cooldown",
    "luck"
]

for stat in shield_stats:
    shield_df[stat + "_score"] = (
        shield_df[stat] - shield_df[stat].min()
    ) / (
        shield_df[stat].max() - shield_df[stat].min()
    )

shield_df.head()

In [ ]:
shield_df[
    [
        "Title",
        "Strength_score",
        "Defence_score",
        "Runic_score",
        "Vitality_score",
        "Cooldown_score",
        "luck_score"
    ]
].head(10)

In [ ]:
def recommend_shield(name):

    weights = get_berserker_weights(name)

    if weights is None:
        return None

    shield_df["Recommendation_Score"] = (
        shield_df["Strength_score"] * weights["Strength"] +
        shield_df["Defence_score"] * weights["Defence"] +
        shield_df["Runic_score"] * weights["Runic"] +
        shield_df["Vitality_score"] * weights["Vitality"] +
        shield_df["Cooldown_score"] * weights["Cooldown"] +
        shield_df["luck_score"] * weights["Luck"]
    )

    result = shield_df.sort_values(
        "Recommendation_Score",
        ascending=False
    )

    return result[
        [
            "Title",
            "Application",
            "Strike",
            "Recommendation_Score"
        ]
    ]

In [ ]:
recommend_shield("King Hrolf Kraki").head(10)

Rönd

In [ ]:
cursor.execute("SELECT * FROM `rönd`")

rond_df = pd.DataFrame(
    cursor.fetchall(),
    columns=[col[0] for col in cursor.description]
)

rond_df.head()

In [ ]:
rond_stats = [
    "Strength",
    "Defence",
    "Runic",
    "Vitality",
    "Cooldown",
    "luck"
]

for stat in rond_stats:
    rond_df[stat + "_score"] = (
        rond_df[stat] - rond_df[stat].min()
    ) / (
        rond_df[stat].max() - rond_df[stat].min()
    )

rond_df.head()

In [ ]:
rond_df[
    [
        "Title",
        "Strength_score",
        "Defence_score",
        "Runic_score",
        "Vitality_score",
        "Cooldown_score",
        "luck_score"
    ]
].head(10)

In [ ]:
def recommend_rond(name):

    weights = get_berserker_weights(name)

    if weights is None:
        return None

    rond_df["Recommendation_Score"] = (
        rond_df["Strength_score"] * weights["Strength"] +
        rond_df["Defence_score"] * weights["Defence"] +
        rond_df["Runic_score"] * weights["Runic"] +
        rond_df["Vitality_score"] * weights["Vitality"] +
        rond_df["Cooldown_score"] * weights["Cooldown"] +
        rond_df["luck_score"] * weights["Luck"]
    )

    result = rond_df.sort_values(
        "Recommendation_Score",
        ascending=False
    )

    return result[
        [
            "Title",
            "Recommendation_Score"
        ]
    ]

test

In [ ]:
recommend_rond("King Hrolf Kraki").head(10)

relics

In [ ]:
cursor.execute("SELECT * FROM relics")

relic_df = pd.DataFrame(
    cursor.fetchall(),
    columns=[col[0] for col in cursor.description]
)

relic_df.head()

In [ ]:
relic_df.columns.tolist()

normalize

In [ ]:
relic_df["Cooldown"] = pd.to_numeric(
    relic_df["Cooldown"],
    errors="coerce"
)

relic_df["Cooldown"] = relic_df["Cooldown"].fillna(0)

relic_df[["Title", "Cooldown"]].head(10)

score

In [ ]:
min_cd = relic_df["Cooldown"].min()
max_cd = relic_df["Cooldown"].max()

if max_cd == min_cd:
    relic_df["Cooldown_score"] = 0
else:
    relic_df["Cooldown_score"] = (
        relic_df["Cooldown"] - min_cd
    ) / (max_cd - min_cd)

relic_df[
    ["Title", "Cooldown", "Cooldown_score"]
].head(10)

In [ ]:
relic_df[["Title", "Effect"]].head(20)

relics recomentdation

In [ ]:
def relic_effect_score(effect, weights):

    if pd.isna(effect):
        return 0

    effect = effect.lower()
    score = 0

    # Strength-related effects
    if "damage" in effect or "melee" in effect:
        score += weights["Strength"] * 0.30

    # Defence / control
    if "stun" in effect or "push" in effect:
        score += weights["Defence"] * 0.25

    # Runic / elemental effects
    if (
        "bifröst" in effect
        or "bifrost" in effect
        or "shock" in effect
        or "realm" in effect
        or "explosion" in effect
        or "seiðr" in effect
    ):
        score += weights["Runic"] * 0.30

    # Rage / survivability
    if "rage" in effect or "health" in effect:
        score += weights["Vitality"] * 0.20

    # Chance-based effects
    if "chance" in effect:
        score += weights["Luck"] * 0.15

    return score

In [ ]:
def recommend_relic(name):

    weights = get_berserker_weights(name)

    if weights is None:
        return None

    relic_df["Recommendation_Score"] = relic_df["Effect"].apply(
        lambda x: relic_effect_score(x, weights)
    )

    result = relic_df.sort_values(
        "Recommendation_Score",
        ascending=False
    )

    return result[
        [
            "Title",
            "Effect",
            "Recommendation_Score"
        ]
    ]

In [ ]:
recommend_relic("King Hrolf Kraki").head(10)

final engine

In [ ]:
def final_build(name):

    # ARMOR
    armor_result = recommendation_details(name)

    # WEAPONS
    axe_result = recommend_axe(name)
    blades_result = recommend_blades(name)
    spear_result = recommend_spear(name)

    # SHIELD
    shield_result = recommend_shield(name)

    # ROND
    rond_result = recommend_rond(name)

    # RELIC
    relic_result = recommend_relic(name)

    if armor_result is None:
        return None

    build = {
        "Berserker": name,

        "Chest": armor_result.loc["Chest", "Title"],
        "Chest_Score": armor_result.loc["Chest", "Recommendation_Score"],

        "Wrist": armor_result.loc["Wrist", "Title"],
        "Wrist_Score": armor_result.loc["Wrist", "Recommendation_Score"],

        "Waist": armor_result.loc["Waist", "Title"],
        "Waist_Score": armor_result.loc["Waist", "Recommendation_Score"],

        "Axe": axe_result.iloc[0]["Title"],
        "Axe_Score": axe_result.iloc[0]["Recommendation_Score"],

        "Blades": blades_result.iloc[0]["Title"],
        "Blades_Score": blades_result.iloc[0]["Recommendation_Score"],

        "Spear": spear_result.iloc[0]["Title"],
        "Spear_Score": spear_result.iloc[0]["Recommendation_Score"],

        "Shield": shield_result.iloc[0]["Title"],
        "Shield_Score": shield_result.iloc[0]["Recommendation_Score"],

        "Rond": rond_result.iloc[0]["Title"],
        "Rond_Score": rond_result.iloc[0]["Recommendation_Score"],

        "Relic": relic_result.iloc[0]["Title"],
        "Relic_Score": relic_result.iloc[0]["Recommendation_Score"]
    }

    return build

test

In [ ]:
build = final_build("King Hrolf Kraki")

build

fix shield

In [ ]:
shield_stats = [
    "Strength",
    "Defence",
    "Runic",
    "Vitality",
    "Cooldown",
    "luck"
]

for stat in shield_stats:

    min_value = shield_df[stat].min()
    max_value = shield_df[stat].max()

    if max_value == min_value:
        shield_df[stat + "_score"] = 0.0
    else:
        shield_df[stat + "_score"] = (
            shield_df[stat] - min_value
        ) / (max_value - min_value)

In [ ]:
build = final_build("King Hrolf Kraki")

build

final

In [ ]:
def print_final_build(name):

    build = final_build(name)

    if build is None:
        return

    print("=" * 60)
    print("       GOD OF WAR RAGNARÖK")
    print("          RECOMMENDED BUILD")
    print("=" * 60)

    print(f"\nBerserker: {build['Berserker']}")
    print("-" * 60)

    print(f"\nARMOR")
    print(f"Chest : {build['Chest']}  ({build['Chest_Score']:.3f})")
    print(f"Wrist : {build['Wrist']}  ({build['Wrist_Score']:.3f})")
    print(f"Waist : {build['Waist']}  ({build['Waist_Score']:.3f})")

    print(f"\nWEAPONS")
    print(f"Axe    : {build['Axe']}  ({build['Axe_Score']:.3f})")
    print(f"Blades : {build['Blades']}  ({build['Blades_Score']:.3f})")
    print(f"Spear  : {build['Spear']}  ({build['Spear_Score']:.3f})")

    print(f"\nDEFENCE")
    print(f"Shield : {build['Shield']}  ({build['Shield_Score']:.3f})")
    print(f"Rönd   : {build['Rond']}  ({build['Rond_Score']:.3f})")

    print(f"\nRELIC")
    print(f"Relic  : {build['Relic']}  ({build['Relic_Score']:.3f})")

    print("\n" + "=" * 60)

In [ ]:
print_final_build("King Hrolf Kraki")

12 all bosses

In [ ]:
all_builds = []

for name in berserker_df["Name"]:

    build = final_build(name)

    if build is not None:
        all_builds.append(build)

all_builds_df = pd.DataFrame(all_builds)

all_builds_df

In [ ]:
all_builds_df[
    [
        "Berserker",
        "Chest",
        "Wrist",
        "Waist",
        "Axe",
        "Blades",
        "Spear",
        "Shield",
        "Rond",
        "Relic"
    ]
]

overallbuld

In [ ]:
score_columns = [
    "Chest_Score",
    "Wrist_Score",
    "Waist_Score",
    "Axe_Score",
    "Blades_Score",
    "Spear_Score",
    "Shield_Score",
    "Rond_Score",
    "Relic_Score"
]

all_builds_df["Build_Score"] = (
    all_builds_df[score_columns].mean(axis=1)
)

all_builds_df[
    ["Berserker", "Build_Score"]
].round(3)

In [ ]:
final_recommendations = all_builds_df[
    [
        "Berserker",
        "Build_Score",
        "Chest",
        "Wrist",
        "Waist",
        "Axe",
        "Blades",
        "Spear",
        "Shield",
        "Rond",
        "Relic"
    ]
].copy()

final_recommendations

create sql table so data may save

In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS final_recommendations (
    ID INT AUTO_INCREMENT PRIMARY KEY,
    Berserker VARCHAR(100),
    Build_Score DECIMAL(6,4),
    Chest VARCHAR(150),
    Wrist VARCHAR(150),
    Waist VARCHAR(150),
    Axe VARCHAR(150),
    Blades VARCHAR(150),
    Spear VARCHAR(150),
    Shield VARCHAR(150),
    Rond VARCHAR(150),
    Relic VARCHAR(150)
)
""")

conn.commit()

print("Final recommendations table created!")

In [ ]:
for _, row in final_recommendations.iterrows():

    cursor.execute("""
        INSERT INTO final_recommendations (
            Berserker,
            Build_Score,
            Chest,
            Wrist,
            Waist,
            Axe,
            Blades,
            Spear,
            Shield,
            Rond,
            Relic
        )
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """, (
        row["Berserker"],
        float(row["Build_Score"]),
        row["Chest"],
        row["Wrist"],
        row["Waist"],
        row["Axe"],
        row["Blades"],
        row["Spear"],
        row["Shield"],
        row["Rond"],
        row["Relic"]
    ))

conn.commit()

print("All final recommendations saved to MySQL!")

In [ ]:
cursor.execute("SELECT * FROM final_recommendations")

saved_recommendations = pd.DataFrame(
    cursor.fetchall(),
    columns=[col[0] for col in cursor.description]
)

saved_recommendations

simple API-style function

In [ ]:
def get_final_recommendation(name):

    result = final_recommendations[
        final_recommendations["Berserker"].str.lower() == name.lower()
    ]

    if result.empty:
        return None

    return result.iloc[0].to_dict()

In [ ]:
get_final_recommendation("King Hrolf Kraki")